In [3]:
import torch
import torch.nn as nn


class InceptionBlock(nn.Module):
    def __init__(
        self,
        in_channels,

        # Branch 1
        out_1x1,

        # Branch 2
        reduce_3x3,
        out_3x3,

        # Branch 3
        reduce_5x5,
        out_5x5,

        # Branch 4
        pool_proj
    ):
        super().__init__()

        # -------------------------
        # Branch 1: 1x1 conv
        # -------------------------
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.ReLU()
        )

        # -------------------------
        # Branch 2: 1x1 -> 3x3
        # -------------------------
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_3x3, kernel_size=1),
            nn.ReLU(),

            nn.Conv2d(
                reduce_3x3,
                out_3x3,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU()
        )

        # -------------------------
        # Branch 3: 1x1 -> 5x5
        # -------------------------
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_5x5, kernel_size=1),
            nn.ReLU(),

            nn.Conv2d(
                reduce_5x5,
                out_5x5,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU()
        )

        # -------------------------
        # Branch 4: pooling -> 1x1
        # -------------------------
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.Conv2d(
                in_channels,
                pool_proj,
                kernel_size=1
            ),
            nn.ReLU()
        )

    def forward(self, x):

        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        print(b1.shape, b2.shape, b3.shape, b4.shape)
        # concatenate along channel dimension
        return torch.cat([b1, b2, b3, b4], dim=1)





In [4]:
# ======================================
# Example
# ======================================

x = torch.randn(1, 192, 28, 28)

model = InceptionBlock(
    in_channels=192,

    out_1x1=64,

    reduce_3x3=96,
    out_3x3=128,

    reduce_5x5=16,
    out_5x5=32,

    pool_proj=32
)

y = model(x)

print(y.shape)

torch.Size([1, 64, 28, 28]) torch.Size([1, 128, 28, 28]) torch.Size([1, 32, 28, 28]) torch.Size([1, 32, 28, 28])
torch.Size([1, 256, 28, 28])


In [5]:
import random

def last_passenger_seat_assignment(total_seats: int) -> int:
    # initialize seats as Boolean list
    seats = [False] * total_seats
    available_seats = list(range(total_seats))
    
    # first passenger chooses a random seat
    p1_seat = random.choice(available_seats)
    seats[p1_seat] = True
    available_seats.remove(p1_seat)

    # subsequent passengers(excluding the last one) choose their seats
    for passenger in range(1, total_seats-1):
        if not seats[passenger]: # if the seat is not occupied
            seats[passenger] = True
            available_seats.remove(passenger)
        else: # if the seat is occupied, pick a random one
            random_seat = random.choice(available_seats)
            seats[random_seat] = True
            available_seats.remove(random_seat)

    # last passenger take the last one remaining seat in available_seats
    last_passenger_seat =  available_seats[0]
    return last_passenger_seat + 1

In [6]:
num_seats = 100
result_seat = last_passenger_seat_assignment(num_seats)
print(f"The 100th passenger sits in seat number: {result_seat}")

The 100th passenger sits in seat number: 1
